In [1]:
# -------------------------------------
# --- Go to correct starting folder ---
# -------------------------------------
# (when running jupyter lab in the browser, the notebook starts with CWD = folder where it is located, which breaks imports, ...)
import os
import pathlib

while not ((cwd := pathlib.Path(os.getcwd())) / "pyproject.toml").exists():
    os.chdir(cwd.parent)  # go 1 folder up

## Approximate log2 over [0.5, 1]  & exp2 over [0, 1] with 2nd degree polynomials

### log2 over [0.5, 1]
- we look for an expression of the form `f(x) ≈ c0 + c1*x + c2*x^2`
- such that `f(0.5) = -1`    (exact fit at lower edge point)
- such that `f(1) = 0`       (exact fit at upper edge point)
- hence `c0=-2 + k1/2, c1=2-1.5*k1, c1=k1` with `k1` a free parameter


### exp2(x) over [0, 1]
- we look for an expression of the form `g(x) ≈ d0 + d1*x + d2*x^2`
- such that `g(0) = 1`    (exact fit at lower edge point)
- such that `g(1) = 2`    (exact fit at upper edge point)
- hence `d0=1, d1=1-k2, d2=k2` with `k2` a free parameter

### jointly optimal

We want to minimize the sum `e_log + e_exp + e_log_exp` where
- `e_log` is maximum error of `f(x)` wrt `log2` over [0.5, 1]
- `e_exp` is maximum error of `g(x)` wrt `exp2` over [0, 1]
- `e_log_exp` is maximum error of `g(f(x)+1)/2` wrt `x` over [0.5, 1]

In [2]:
import math

import numpy as np

from max_div.internal.math.fast_pow import construct_calibration_data

In [3]:
# -------------------------------------------------------------------------
#  log2
# -------------------------------------------------------------------------
def compute_c(_k1: float) -> list[float]:
    return [-2 + 0.5 * _k1, 2 - 1.5 * _k1, _k1]


def log2_approx(x: float, c: list[float]) -> float:
    """Only efficient for x in or close to [0.5, 1]"""
    if x <= 0:
        return -1e6
    if x < 0.5:
        return log2_approx(2 * x, c) - 1
    elif x > 1:
        return log2_approx(x / 2, c) + 1
    else:
        # f(x) - but only valid in [0.5, 1]
        return c[0] + (c[1] * x) + (c[2] * x * x)


# -------------------------------------------------------------------------
#  exp2
# -------------------------------------------------------------------------
def compute_d(_k2: float) -> list[float]:
    return [1.0, 1 - _k2, _k2]


def exp2_approx(x: float, d: list[float]) -> float:
    """Only efficient for x in or close to [0, 1]"""
    if x < 0:
        return exp2_approx(x + 1, d) / 2
    elif x > 1:
        return exp2_approx(x - 1, d) * 2
    else:
        # g(x) - but only valid in [0, 1]
        return d[0] + (d[1] * x) + (d[2] * x * x)


# -------------------------------------------------------------------------
#  pow
# -------------------------------------------------------------------------
def pow_approx(x: float, t: float, c: list[float], d: list[float]) -> float:
    """Approximate x^t = exp2_approx(t*log2_approx(x))"""
    return exp2_approx(t * log2_approx(x, c), d)


# -------------------------------------------------------------------------
#  Overall cost function
# -------------------------------------------------------------------------
def cost_function(_k1: float, _k2: float, _n: int) -> float:
    # get calibration data  (below function is lru_cached, so fast after first call)
    _x_arr, _t_arr, _xt_exact_arr = construct_calibration_data(_n)

    # coefficients
    c = compute_c(_k1)
    d = compute_d(_k2)

    # sample points
    deltas = np.array(
        [_xt_exact - pow_approx(_x, _t, c, d) for _x, _t, _xt_exact in zip(_x_arr, _t_arr, _xt_exact_arr)]
    )

    # compute error
    # error = math.sqrt(np.mean(deltas*deltas))   # RMSE
    error = np.max(np.abs(deltas))  # max abs error
    return error

In [4]:
optimal_k1 = -1.333
optimal_k2 = 0.333

# for i in range(60):
for ii in range(110):
    # Create grid
    # grid_size = 0.5**i
    grid_size = 0.7**ii
    k1_values = np.linspace(optimal_k1 - grid_size, optimal_k1 + grid_size, 10)
    k2_values = np.linspace(optimal_k2 - grid_size, optimal_k2 + grid_size, 10)

    # Evaluate cost function at each grid point
    optimal_cost = 1e12
    optimal_k1 = 1e3
    optimal_k2 = 1e3
    for i, k1 in enumerate(k1_values):
        for j, k2 in enumerate(k2_values):
            cost = cost_function(k1, k2, 100)
            if cost < optimal_cost:
                optimal_cost = cost
                optimal_k1 = k1
                optimal_k2 = k2

    print(ii, grid_size, optimal_cost, f"{optimal_k1:.20f}", f"{optimal_k2:.20f}")

0 1.0 0.018763905312574103 -0.99966666666666692542 0.22188888888888880224
1 0.7 0.014086747111994646 -1.23300000000000009592 0.29966666666666663676
2 0.48999999999999994 0.011625898010462266 -1.17855555555555557845 0.35411111111111098770
3 0.3429999999999999 0.010754528375853223 -1.21666666666666678509 0.31599999999999989209
4 0.24009999999999995 0.010587229945281551 -1.18998888888888898485 0.34267777777777763681
5 0.16806999999999994 0.009169733733949359 -1.20866333333333342281 0.32400333333333319885
6 0.11764899999999996 0.009382792709017318 -1.22173544444444459600 0.33707544444444431653
7 0.08235429999999996 0.00907967302819096 -1.21258496666666681918 0.32792496666666653971
8 0.05764800999999997 0.009170984436748353 -1.21899030111111117414 0.33433030111111095017
9 0.04035360699999998 0.00907816672559364 -1.21450656700000014787 0.32984656699999981289
10 0.02824752489999998 0.00908058481678542 -1.21764518087777795508 0.33298518087777762009
11 0.019773267429999988 0.00907750256107831 -

In [5]:
# --- log2 ------------------------------------------------
c_opt = compute_c(float(optimal_k1))
print("log2 approx coefficients:", c_opt)
# max_error_log2 = cost_function_log2(optimal_k1, 100)
# print("max abs error log2 approx over [0.5, 1]:", max_error_log2)
# print()

# --- exp2 ------------------------------------------------
d_opt = compute_d(float(optimal_k2))
print("exp2 approx coefficients:", d_opt)
# max_error_exp2 = cost_function_exp2(optimal_k2, 100)
# print("max rel error exp2 approx over [0, 1]:", max_error_exp2)
# print()

# --- combined --------------------------------------------
# max_error_log2_exp2 = _cost_function_log2_exp2(optimal_k1, optimal_k2, 10000)
# max_error_exp2_log2 = _cost_function_exp2_log2(optimal_k1, optimal_k2, 10000)
#
# print("max abs error exp2(log2(x)) approx over [0.5, 1]:", max_error_log2_exp2)
# print("max abs error log2(exp2(x)) approx over [0.0, 1]:", max_error_exp2_log2)

log2 approx coefficients: [-2.6080969294048635, 3.8242907882145905, -1.216193858809727]
exp2 approx coefficients: [1.0, 0.6700844332949878, 0.32991556670501215]


In [6]:
0.7**100
0.5**60

8.673617379884035e-19